In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

from pixell import enmap, enplot, curvedsky, wcsutils
import healpy as hp
from mnms import utils, tiled_noise

tubes = [
    'lat_f090',
    'lat_f150',
    'lat_f220',
    'lat_f280'
]

In [ ]:
# prep noise models. we make three changes from dr6

shift_by_band = {
    'lat_f030': -300,
    'lat_f090': 0,
    'lat_f220': 1.5, # lookback factor in ell
}

band_pairs = {
    'lat_f030': 'lat_f040',
    'lat_f090': 'lat_f150',
    'lat_f220': 'lat_f280'
}

ref_models = {
    'lat_f030': 'pa5_f090_pa5_f150',
    'lat_f090': 'pa5_f090_pa5_f150',
    'lat_f220': 'pa4_f150_pa4_f220'
}

for k in tubes:
    if k in band_pairs:
        ref_model = ref_models[k]

        # the cov ells in the act noise models are after appropriate
        # sqrt_ivar flattening (eg, so dg4 cov ell is exactly 4 times dg2). so 
        # all of the depth stuff is captured in the ivars we just made. 

        # first, we need to rescale the cov ells a little bit so they match onto the
        # ivar level at high ell. we'll do the correction using the last value of the 
        # dg2 TT spectra. because of the pixwin tilt, the correction is OK for 
        # intermediate-ell pol spectra, but it's a little high for high-ell. this can
        # be corrected on-the-fly with the sqrt_cov_ell ratio between pol and T

        sqrt_cov_mat, _, extra_datasets_dict = tiled_noise.read_tiled_ndmap(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/dr6v4/simulations/noise/models/act_dr6v4_tile_cmbmask_{ref_model}_lmax10800_4way_set0_noise_model.hdf5',
                                                                            extra_datasets=['sqrt_cov_ell'])
        scedg2 = extra_datasets_dict['sqrt_cov_ell']
        cedg2 = scedg2.reshape(6, 6, -1)
        cedg2 = utils.eigpow(cedg2, 2, axes=[0, 1])

        # the correction is independent of the absolute ivar level (but not the mask downgrade;
        # this exact notebook works because for a dg2 noise model with postfiltreldg, the actual
        # sqrtcovell was measured with dg1, and then the whitecedg is calced for the same) because the cov ells
        # are flattened, so we just assume std normal noise (this is true for the 
        # noise we are trying to match onto, which will be std normal after ivar flatt-
        # ening). NOTE: this also assumes the tiled part of the model returns ~1

        # only works for MF and UHF
        shape, wcs = enmap.read_map_geometry('/home/zatkins/scratch/projects/lat-iso/piso/misc/piso_sims/so/footprint/so_geometry_deep56_v20250306_lat_f090.fits')

        white_cedg = np.sum(1**2*enmap.pixsizemap(shape, wcs) * enmap.pixsizemap(shape, wcs))/(4*np.pi) # white noise pseudospectrum in patch
        white_cedg /= np.sum(1**2 * enmap.pixsizemap(shape, wcs))/(4*np.pi) # w2 of patch
        fac = np.zeros((6, 6, 1))

        fac[0, 0] = white_cedg / cedg2[0, 0, -1]
        fac[1, 1] = fac[0, 0]
        fac[2, 2] = fac[0, 0]
        fac[3, 3] = white_cedg / cedg2[3, 3, -1]
        fac[4, 4] = fac[3, 3]
        fac[5, 5] = fac[3, 3]

        for i in range(6):
            for j in range(6):
                if i == j:
                    continue
                fac[i, j] = np.sqrt(fac[i, i] * fac[j, j]) # keep the corr constant

        print(fac.squeeze())
        cedg2 *= fac

        # next, we want to shift the power spectra left and right depending on the band
        # of the tube (shift right for uhf, left for lf). we do this literally moving the
        # spectrum to the left for lf, but for uhf we rescale the spectrum. note we
        # assume bands pair-off per tube like dr6, so we just list this by the first
        # band in the tube

        shift = shift_by_band[k]
        band_pair = (k, band_pairs[k])

        if shift < 0:
            this_cedg2 = np.zeros_like(cedg2)
            
            for i in range(6):
                for j in range(6):
                    seq_of_arrs = (cedg2[i, j, :2], cedg2[i, j, shift+2:], cedg2[i, j, -1]*np.ones(shift, dtype=cedg2.dtype))
                    this_cedg2[i, j] = np.concatenate(seq_of_arrs)
        
        elif shift > 0:
            this_cedg2 = np.zeros_like(cedg2)
            out_l = np.arange(this_cedg2.shape[-1])
            in_l = np.round(out_l / shift).astype(int)
            
            for i in range(6):
                for j in range(6):
                    this_cedg2[i, j, out_l] = cedg2[i, j, in_l]

        else:
            this_cedg2 = cedg2

        scedg2 = utils.eigpow(this_cedg2, 0.5, axes=[0, 1])
        scedg2 = scedg2.reshape(2, 1, 3, 2, 1, 3, -1)

        # finally we don't need all the tiles, just the ones in our patch. so, for each
        # tile in the patch, find the tile in dr6 that is closest and just use that 
        # data. NOTE: because tiled_noise uses "phys" noramlization, and we are 
        # changing the patch, we need to adjust for the new "phys" normalization
        shapedg2, wcsdg2 = enmap.downgrade_geometry(shape, wcs, 2)
        sqm = enmap.zeros((6, 6, *shapedg2), wcsdg2, dtype=np.float32)
        sqm = tiled_noise.tiled_ndmap(sqm)
        sqm = sqm.to_tiled()
        sqm = sqm[..., :sqm.shape[-1]//2 + 1]

        phys_fac = (sqm.pixsize() / sqrt_cov_mat.pixsize())**0.5

        for o, otile_idx in enumerate(range(sqm.num_tiles)):
            oshape, owcs = sqm.get_tile_geometry(otile_idx)
            ref = enmap.pix2sky(oshape, owcs, [[0], [0]])

            # get closest tile
            neardist = np.inf
            neari = -1
            nearitile_idx = -1
            for i, itile_idx in enumerate(sqrt_cov_mat.unmasked_tiles):
                ishape, iwcs = sqrt_cov_mat.get_tile_geometry(itile_idx)
                test = enmap.pix2sky(ishape, iwcs, [[0], [0]])
                test_dist = (test[0, 0] - ref[0, 0])**2 + (test[1, 0] - ref[1, 0])**2

                if test_dist < neardist:
                    neardist = test_dist
                    neari = i
                    nearitile_idx = itile_idx

            print(o, otile_idx, neari, nearitile_idx)
            sqm[o] = sqrt_cov_mat[neari] * phys_fac

        # now save the noise models
        fn = '/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/so/simulations/noise/models/'
        tiled_noise.write_tiled_ndmap(fn + f"lat_piso_tile_cmbmask_{'_'.join(band_pair)}_lmax10800_4way_setX_noise_model.hdf5",
                                    sqm,
                                    extra_datasets={'sqrt_cov_ell': scedg2})

In [ ]:
# prep beams and leakage

# we use gaussian beams from the aso paper, to the lmax of the dr6 beams
# we use sqrt(3)x the dr6 error modes (3x at covariance level)
beam_dr6 = np.loadtxt('/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/dr6v4/beams/coadd_pa5_f090_night_beam_tform_jitter_cmb.txt')
rel_beam_dr6_error_modes = beam_dr6[:, 2:] / beam_dr6[:, 1][:, None] # normalize by b_l
rel_beam_dr6_error_modes = rel_beam_dr6_error_modes * 1.75
l = np.arange(rel_beam_dr6_error_modes.shape[0])

fwhm_by_band = { # arcmin
    'lat_f030': 7.4,
    'lat_f040': 5.1,
    'lat_f090': 2.2,
    'lat_f150': 1.4,
    'lat_f220': 1.0,
    'lat_f280': 0.9    
}

for k in fwhm_by_band:
    bl = hp.gauss_beam(np.deg2rad(fwhm_by_band[k]/60), lmax=l[-1])
    beam = np.zeros((l.size, rel_beam_dr6_error_modes.shape[1]+2))
    beam[:, 0] = l
    beam[:, 1] = bl
    beam[:, 2:] = rel_beam_dr6_error_modes * bl[:, None]
    np.savetxt(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/so/beams/main_beams/beam_{k}.txt', beam)

# we use the same leakage template as dr6_pa5_f090, but rescaled by a variable
# amount (including error modes) depending on the tube, so that null tests are
# non-trivial
tubes_to_facs = {
    'c1': -0.5,
    'i1': .75,
    'i3': -1,
    'i4': 1.25,
    'i5': -1.5,
    'i6': 1.75,
    'o6': -2
}

for p in ['e', 'b']:
    leakage = np.loadtxt(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/dr6v4/beams/pa5_f090_gamma_t2{p}.txt')
    for k, fac in tubes_to_facs.items():
        _leakage = leakage.copy()
        _leakage[:, 1:] = leakage[:, 1:] * fac
        np.savetxt(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/so/beams/leakage_beams/gamma_t2{p}_{k}.txt', _leakage)